# Taoyuan Canonical Elastic Model —— OpenSeesPy 版本(第一塊地基)

延續「先小後大」原則,鎖定完整桃園 `8` 柱建築裡的**其中一榀 `Y` 向
`1` 跨 `2` 柱、`2` 層構架**——不是完整 `8` 柱建築。

**這不是延續 `Case-06.5`/`Case-06.6`**,是全新的一條主線:

- 真梁真柱(不用 `Case-03` 系列的剪力構架簡化,不用 `ops.fix` 拘束轉角)
- 斷面勁度用**規範標準折減係數**(梁 `0.35Ig`、柱 `0.7Ig`),$I_g$ 本身
  刻意不計入鋼筋(`ACI 318` 委員會註解逐字定義),不需要先假設任何
  鋼筋比例
- 斷面尺寸(柱 `40x40cm`、梁 `30x50cm`)明確標記為**這個案例的初步
  試設**,沿用歷史數字只是為了延續性,不是宣稱這是「最終答案」或
  「業界底線」
- 目標:算出每根梁柱各自真正的 $P_u/M_u/V_u$,並記錄是哪個載重組合
  (`governing combination`)產生的——不是把不同工況的最大值硬湊成
  一個不存在的組合

`OpenSeesPy` 先做扎實,通過平衡驗證之後,才疊加 `PyFEM`/`PyNite` 做
跨工具比較(下一步)。

## 基本參數——任何人可以拿這組資料用自己的工具重新建模對照

In [1]:

import openseespy.opensees as ops
import numpy as np

# ============================================================
# 幾何
# ============================================================
L_bay = 6.0      # 跨度 (m)
h1 = 3.5         # 1F樓高 (m)
h2 = 3.5         # 2F樓高 (m)

# ============================================================
# 斷面(初步試設, 不是最終設計答案)
# ============================================================
h_col = 0.40     # 柱: 40x40cm 正方形
b_beam, h_beam = 0.30, 0.50   # 梁: 寬30cm x 深50cm

# ============================================================
# 材料
# ============================================================
E_rc = 2.463e7   # 混凝土彈性模數 (kPa), 對應fc'=280kgf/cm^2

# ============================================================
# 規範標準勁度折減(不含鋼筋, Ig本身就是"neglecting reinforcement")
# ============================================================
Ig_col = h_col**4/12
Ig_beam = b_beam*h_beam**3/12
Ic = 0.7*Ig_col     # 柱: 0.7Ig
Ib = 0.35*Ig_beam   # 梁: 0.35Ig
Ac = h_col**2
Ab = b_beam*h_beam

print(f"柱: 40x40cm, Ig={Ig_col:.6e}m^4, 折減後Ic(0.7Ig)={Ic:.6e}m^4")
print(f"梁: 30x50cm, Ig={Ig_beam:.6e}m^4, 折減後Ib(0.35Ig)={Ib:.6e}m^4")
print(f"E_rc = {E_rc:.4e} kPa")

# ============================================================
# 載重
# ============================================================
P_2F = 147.60        # kN, 屋頂樓層每根柱頂承受的重力軸力
P_1F_extra = 147.60  # kN, 2F樓板每根柱頂額外承受的重力軸力
F1_Y, F2_Y = 9.938, 15.900   # kN, 1F/2F樓層等效側力(Y方向規範地震力)

print(f"\n重力: 2F樓層每柱{P_2F}kN, 屋頂樓層每柱{P_1F_extra}kN")
print(f"側力: F1_Y={F1_Y}kN(施加於2F樓層), F2_Y={F2_Y}kN(施加於屋頂樓層)")


柱: 40x40cm, Ig=2.133333e-03m^4, 折減後Ic(0.7Ig)=1.493333e-03m^4
梁: 30x50cm, Ig=3.125000e-03m^4, 折減後Ib(0.35Ig)=1.093750e-03m^4
E_rc = 2.4630e+07 kPa

重力: 2F樓層每柱147.6kN, 屋頂樓層每柱147.6kN
側力: F1_Y=9.938kN(施加於2F樓層), F2_Y=15.9kN(施加於屋頂樓層)


## 模型建構——節點/元素定義

In [2]:

ELEMENT_ORIENTATION = {1:'vertical', 2:'vertical', 3:'vertical', 4:'vertical',
                        5:'horizontal', 6:'horizontal'}
ELEMENT_NAMES = {1:'1F左柱', 2:'1F右柱', 3:'2F左柱', 4:'2F右柱', 5:'1F樑', 6:'屋頂樑'}

def build_real_frame():
    ops.wipe(); ops.model('basic', '-ndm', 2, '-ndf', 3)
    # 節點: 1,2=地面(固定支撐); 3,4=1F樓層(2F樓板高度); 5,6=屋頂樓層
    ops.node(1, 0.0, 0.0);   ops.node(2, L_bay, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L_bay, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L_bay, h1+h2)
    ops.fix(1, 1, 1, 1); ops.fix(2, 1, 1, 1)
    ops.geomTransf('Linear', 1)
    # 元素1-4: 柱(1F左/右, 2F左/右); 元素5-6: 梁(1F樓板梁, 屋頂梁)
    ops.element('elasticBeamColumn', 1, 1, 3, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 2, 2, 4, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 3, 3, 5, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 4, 4, 6, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 5, 3, 4, Ab, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 6, 5, 6, Ab, E_rc, Ib, 1)

build_real_frame()
print("模型建構成功:", len(ops.getNodeTags()), "節點,", len(ops.getEleTags()), "元素")


模型建構成功: 6 節點, 6 元素


## 求解跟內力萃取——重要的建模陷阱,誠實記錄

**這裡真的踩過一次坑,不是順利做完**:`OpenSeesPy` 的 `eleForce()`
回傳的是**全域座標系**下的 `[Fx,Fy,M]`,不是局部座標系的 `[N,V,M]`。
柱子垂直放置、梁水平放置,兩者的「軸力」跟「剪力」對應到全域
`Fx`/`Fy` 的方式剛好相反——第一次沒注意到這件事,算出來的 $V_u$
（`281.55kN`)量級離譜地大,才發現搞反了。用一個答案已知的簡單懸臂
案例(純軸力 `vs` 純剪力各自測試)驗證過index順序之後才修正。

In [3]:

def run_case(loads):
    """loads: list of (node, Fx, Fy, Mz). 回傳每個元素各端的[N,V,M],
    已經根據元素方向(垂直/水平)正確對應到局部軸力/剪力。"""
    build_real_frame()
    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    for (n, fx, fy, mz) in loads:
        ops.load(n, fx, fy, mz)
    ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
    ops.algorithm('Linear'); ops.integrator('LoadControl', 1.0); ops.analysis('Static')
    ops.analyze(1)
    ops.reactions()
    results = {}
    for eid in [1,2,3,4,5,6]:
        f = ops.eleForce(eid)  # 全域座標系 [Fxi,Fyi,Mi, Fxj,Fyj,Mj]
        if ELEMENT_ORIENTATION[eid] == 'vertical':
            Ni, Vi, Mi = f[1], f[0], f[2]
            Nj, Vj, Mj = f[4], f[3], f[5]
        else:
            Ni, Vi, Mi = f[0], f[1], f[2]
            Nj, Vj, Mj = f[3], f[4], f[5]
        results[eid] = dict(Ni=Ni, Vi=Vi, Mi=Mi, Nj=Nj, Vj=Vj, Mj=Mj)
    # 額外記錄支撐反力(最直接、不需要透過元素內力正負號換算的確認方式)
    results['_reactions'] = {1: ops.nodeReaction(1), 2: ops.nodeReaction(2)}
    return results

# Case D+L: 純重力
gravity_loads = [(3,0,-P_1F_extra,0), (4,0,-P_1F_extra,0), (5,0,-P_2F,0), (6,0,-P_2F,0)]
R_grav = run_case(gravity_loads)

# Case E: 純側推(規範等效地震力, Y方向)
lateral_loads = [(3,F1_Y,0,0), (5,F2_Y,0,0)]
R_lat = run_case(lateral_loads)

print("兩個獨立load case求解完成: D+L(重力), E(側推)")


兩個獨立load case求解完成: D+L(重力), E(側推)


## 平衡檢核——先確認模型本身沒有建模錯誤,再談後續

這是最基本、最重要的健全性檢查:總反力必須精確等於施加的總外力,
沒有任何近似、沒有容差空間。

In [4]:

# 平衡檢核改用"支撐反力"直接驗證——這是最直接、不需要透過元素內力
# 正負號換算的方式(之前用元素Ni組合時, 曾經因為多加了一個不必要的
# 負號, 導致assert失敗, 這裡改用reactions()直接讀支撐反力來源, 更穩健)
Rx1, Ry1, Mz1 = R_grav['_reactions'][1]
Rx2, Ry2, Mz2 = R_grav['_reactions'][2]
N_base_total_grav = Ry1 + Ry2

print(f"[平衡檢核1] 重力總支撐反力(Fy) = {N_base_total_grav:.4f}kN")
print(f"            應等於施加總重力 = {2*P_1F_extra+2*P_2F:.4f}kN")
assert abs(N_base_total_grav - (2*P_1F_extra+2*P_2F)) < 1e-6, "重力平衡檢核失敗"

Rx1_l, Ry1_l, Mz1_l = R_lat['_reactions'][1]
Rx2_l, Ry2_l, Mz2_l = R_lat['_reactions'][2]
V_base_total_lat = Rx1_l + Rx2_l

print(f"\n[平衡檢核2] 側推總支撐反力(Fx) = {V_base_total_lat:.4f}kN")
print(f"            應等於施加總側力 = {F1_Y+F2_Y:.4f}kN")
assert abs(abs(V_base_total_lat) - (F1_Y+F2_Y)) < 1e-6, "側推平衡檢核失敗(支撐反力方向本來就跟施加力相反, 比較大小不比較符號)"

print("\n[PASS] 兩個平衡檢核都精確通過(不是近似), 用支撐反力直接驗證,")
print("不透過元素內力正負號換算, 模型本身沒有建模錯誤")


[平衡檢核1] 重力總支撐反力(Fy) = 590.4000kN
            應等於施加總重力 = 590.4000kN

[平衡檢核2] 側推總支撐反力(Fx) = -25.8380kN
            應等於施加總側力 = 25.8380kN

[PASS] 兩個平衡檢核都精確通過(不是近似), 用支撐反力直接驗證,
不透過元素內力正負號換算, 模型本身沒有建模錯誤


## 載重組合跟 Governing Demand——每個數字都連著它的來源

不是把不同工況的最大值各自獨立取出湊成一個不存在的組合,是針對
`D+L+E`/`D+L-E` 這兩個真實存在的組合,各自算出完整的 $(P,V,M)$,
再取「以 $|M|$ 為準」的那一個當作這個端點的 governing demand,並且
**保留是哪個組合產生的**。

In [5]:

def get_governing_table():
    table = []
    for eid in [1,2,3,4,5,6]:
        for end in ['i','j']:
            N_g, V_g, M_g = R_grav[eid][f'N{end}'], R_grav[eid][f'V{end}'], R_grav[eid][f'M{end}']
            N_l, V_l, M_l = R_lat[eid][f'N{end}'], R_lat[eid][f'V{end}'], R_lat[eid][f'M{end}']
            combos = {'D+L+E': (N_g+N_l, V_g+V_l, M_g+M_l),
                      'D+L-E': (N_g-N_l, V_g-V_l, M_g-M_l)}
            gov_combo = max(combos, key=lambda k: abs(combos[k][2]))
            N, V, M = combos[gov_combo]
            table.append(dict(member=ELEMENT_NAMES[eid], end=end, combo=gov_combo,
                               Pu=abs(N), Vu=abs(V), Mu=abs(M)))
    return table

governing_table = get_governing_table()

print(f"{'構件':<8}{'端':<4}{'governing':<10}{'Pu(kN)':>10}{'Vu(kN)':>10}{'Mu(kN-m)':>12}")
print('-'*56)
for r in governing_table:
    print(f"{r['member']:<8}{r['end']:<4}{r['combo']:<10}{r['Pu']:>10.2f}{r['Vu']:>10.2f}{r['Mu']:>12.2f}")


構件      端   governing     Pu(kN)    Vu(kN)    Mu(kN-m)
--------------------------------------------------------
1F左柱    i   D+L+E         281.55     12.94       32.13
1F左柱    j   D+L+E         281.55     12.94       13.16
1F右柱    i   D+L+E         308.85     12.90       32.03
1F右柱    j   D+L+E         308.85     12.90       13.11
2F左柱    i   D+L+E         142.01      7.95       11.03
2F左柱    j   D+L+E         142.01      7.95       16.78
2F右柱    i   D+L+E         153.19      7.95       11.06
2F右柱    j   D+L+E         153.19      7.95       16.78
1F樑     i   D+L+E           4.95      8.06       24.19
1F樑     j   D+L+E           4.95      8.06       24.17
屋頂樑     i   D+L+E           7.95      5.59       16.78
屋頂樑     j   D+L+E           7.95      5.59       16.78


## 小結

- 建立了完整桃園建築裡 `Y` 向 `1` 跨 `2` 柱構架的真梁真柱線彈性模型,
  用規範標準勁度折減(`0.35Ig`/`0.7Ig`),不需要先假設任何鋼筋比例
- 兩個平衡檢核(重力總反力、側推總柱底剪力)都精確通過,確認模型
  本身沒有建模錯誤
- **誠實記錄一個真實踩過的坑**:`eleForce()` 回傳全域座標系內力,
  柱(垂直)跟梁(水平)的軸力/剪力對應到全域 `Fx`/`Fy` 的方式相反,
  第一次沒注意到導致 `Vu` 算出離譜的數字,用已知答案的簡單案例驗證
  index順序之後才修正
- 完整的 `governing demand` 表格,每個數字都連著產生它的載重組合
  (`D+L+E`/`D+L-E`),不是把不同工況的最大值硬湊
- **這是這條新主線的第一塊地基**,下一步:同一組幾何/材料/載重,
  疊加 `PyFEM`/`PyNite`,驗證三個工具是否對得上同一個答案,再拿其中
  一套當正式設計需求來源

## 跨工具驗證:`frame2d`(使用者自建的矩陣位移法框架)

用完全相同的幾何/材料/斷面/載重,拿到 `frame2d-matrix-analysis` 這個
獨立開發的框架重新跑一次,比對是否對得上同一個答案。

**過程中先踩過一個坑,不是一次就對上**:第一次比對時,`OpenSeesPy`
跟 `frame2d` 的答案有 `0.03%~0.46%` 差異——比線彈性理論上該有的
誤差量級大。查證後發現:給對方的參數摘要表裡只列了慣性矩 $I$,
**漏了軸向面積 $A$**,對方只能自己設一個值(用了 `100000cm²`,遠遠
超過正確值 `1600`/`1500cm²`,等於把構件軸向設成近乎剛性)。補上
正確的 $A$ 值重新比對後,差異才收斂到 `<0.02%`。

In [6]:

# frame2d的原始輸出(獨立求解器算出的結果, 不是這裡即時執行——
# frame2d是使用者自己在另一個repo運行的, 這裡只是把它的輸出記錄
# 下來做比對, 這組數字用的是跟OpenSeesPy完全一致的A=1600/1500cm^2)
frame2d_results = {
    ('1F左柱','i'): dict(Pu=281.546, Vu=12.939, Mu=32.129),
    ('1F左柱','j'): dict(Pu=281.546, Vu=12.939, Mu=13.159),
    ('1F右柱','i'): dict(Pu=308.854, Vu=12.899, Mu=32.033),
    ('1F右柱','j'): dict(Pu=308.854, Vu=12.899, Mu=13.112),
    ('2F左柱','i'): dict(Pu=142.006, Vu=7.947,  Mu=11.031),
    ('2F右柱','i'): dict(Pu=153.194, Vu=7.953,  Mu=11.057),
    ('1F樑','i'):   dict(Pu=4.946,   Vu=8.060,  Mu=24.190),
    ('屋頂樑','i'):  dict(Pu=7.953,   Vu=5.594,  Mu=16.784),
}

print(f"{'構件/端':<12}{'項目':<6}{'OpenSeesPy':>12}{'frame2d':>12}{'差異%':>10}")
print('-'*54)
max_diff = 0.0
opensees_lookup = {}
for eid in [1,2,3,4,5,6]:
    for end in ['i','j']:
        Ng,Vg,Mg = R_grav[eid][f'N{end}'], R_grav[eid][f'V{end}'], R_grav[eid][f'M{end}']
        Nl,Vl,Ml = R_lat[eid][f'N{end}'], R_lat[eid][f'V{end}'], R_lat[eid][f'M{end}']
        opensees_lookup[(ELEMENT_NAMES[eid], end)] = dict(
            Pu=abs(Ng+Nl), Vu=abs(Vg+Vl), Mu=abs(Mg+Ml))

for key, ref in frame2d_results.items():
    mine = opensees_lookup[key]
    for item in ['Pu','Vu','Mu']:
        diff = abs(mine[item]-ref[item])/abs(ref[item])*100 if ref[item] != 0 else 0
        max_diff = max(max_diff, diff)
        print(f"{key[0]:<8}{key[1]:<4}{item:<6}{mine[item]:>12.4f}{ref[item]:>12.4f}{diff:>10.4f}")

print(f"\n最大差異: {max_diff:.4f}%")
assert max_diff < 0.02, "跟frame2d的差異應該在0.02%以內(用同一組A值後)"
print("[PASS] OpenSeesPy跟frame2d(完全獨立的兩個求解器)在同一組輸入下,")
print("答案幾乎完全一致——這是這條新主線第一次真正的跨工具交叉驗證")


構件/端        項目      OpenSeesPy     frame2d       差異%
------------------------------------------------------
1F左柱    i   Pu        281.5470    281.5460    0.0004
1F左柱    i   Vu         12.9394     12.9390    0.0029
1F左柱    i   Mu         32.1304     32.1290    0.0044
1F左柱    j   Pu        281.5470    281.5460    0.0004
1F左柱    j   Vu         12.9394     12.9390    0.0029
1F左柱    j   Mu         13.1574     13.1590    0.0121
1F右柱    i   Pu        308.8530    308.8540    0.0003
1F右柱    i   Vu         12.8986     12.8990    0.0029
1F右柱    i   Mu         32.0346     32.0330    0.0051
1F右柱    j   Pu        308.8530    308.8540    0.0003
1F右柱    j   Vu         12.8986     12.8990    0.0029
1F右柱    j   Mu         13.1105     13.1120    0.0111
2F左柱    i   Pu        142.0064    142.0060    0.0003
2F左柱    i   Vu          7.9473      7.9470    0.0034
2F左柱    i   Mu         11.0312     11.0310    0.0021
2F右柱    i   Pu        153.1936    153.1940    0.0003
2F右柱    i   Vu          7.9527      7.9530  

## 左右柱剪力微小不對稱——真實的物理效應,不是誤差

比對過程中發現一個值得深究的細節:`1F` 左柱跟右柱的 $V_u$(`12.94`
`vs` `12.90`)**不完全相等**,即使兩根柱子斷面、材料完全相同。純
水平力作用下(無扭轉/偏心),理論上兩根柱子該分擔相等剪力——這個
微小不對稱哪來的?

用不同軸向面積 $A$ 重新測試,確認了根因:**梁的軸向勁度不是無限大**
(這裡用真實的 $A_b=1500cm^2$,不是近乎剛性的假設值),讓側推力
透過梁傳遞到對側柱子時,梁本身有一點點軸向變形,導致左右柱位移/
內力不完全對稱。軸向勁度越趨近無限大(樓板越接近完全剛性隔板),
這個不對稱就越小——這不是誤差或 `bug`,是「樓板剛性隔板」這個
建模假設本身帶來的真實效應,**兩個完全獨立的求解器(`OpenSeesPy`
跟 `frame2d`)都精確捕捉到了同一個現象**(差異對比表裡兩邊的左右
柱不對稱幅度幾乎一致),這正好是這次交叉驗證最有力的證據之一。

In [7]:

# 驗證: 軸向勁度趨近無限大時, 左右柱剪力差異趨近於零
def test_asymmetry(Ac_test, Ab_test, label):
    ops.wipe(); ops.model('basic', '-ndm', 2, '-ndf', 3)
    ops.node(1, 0.0, 0.0);   ops.node(2, L_bay, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L_bay, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L_bay, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)
    ops.geomTransf('Linear', 1)
    ops.element('elasticBeamColumn', 1, 1, 3, Ac_test, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 2, 2, 4, Ac_test, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 3, 3, 5, Ac_test, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 4, 4, 6, Ac_test, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 5, 3, 4, Ab_test, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 6, 5, 6, Ab_test, E_rc, Ib, 1)
    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    ops.load(3,F1_Y,0,0); ops.load(5,F2_Y,0,0)
    ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
    ops.algorithm('Linear'); ops.integrator('LoadControl',1.0); ops.analysis('Static')
    ops.analyze(1)
    f1 = ops.eleForce(1); f2 = ops.eleForce(2)
    diff = abs(f1[0]-f2[0])
    print(f"{label}: 1F左柱Vu={abs(f1[0]):.4f}, 1F右柱Vu={abs(f2[0]):.4f}, 差異={diff:.6f}kN")
    return diff

d1 = test_asymmetry(Ac, Ab, "真實A值(1600/1500cm^2)")
d2 = test_asymmetry(10.0, 10.0, "近乎無限剛(100000cm^2, 對照組)")
d3 = test_asymmetry(1000.0, 1000.0, "極端趨近無限剛")

assert d1 > d2 > d3, "軸向勁度越大, 左右柱剪力不對稱應該越小"
print("\n[PASS] 確認左右柱剪力不對稱隨軸向勁度增加而收斂到零,")
print("這是真實的樓板剛性隔板效應, 不是求解誤差")


真實A值(1600/1500cm^2): 1F左柱Vu=12.9394, 1F右柱Vu=12.8986, 差異=0.040755kN
近乎無限剛(100000cm^2, 對照組): 1F左柱Vu=12.9193, 1F右柱Vu=12.9187, 差異=0.000616kN
極端趨近無限剛: 1F左柱Vu=12.9190, 1F右柱Vu=12.9190, 差異=0.000006kN

[PASS] 確認左右柱剪力不對稱隨軸向勁度增加而收斂到零,
這是真實的樓板剛性隔板效應, 不是求解誤差


## 跨工具驗證:`PyNite`

同一組幾何/材料/斷面/載重,換算成 `PyNite` 慣用的 `N-mm` 制,3D 框架
限制平面外自由度,做成等效的 2D 分析。

In [8]:

try:
    from Pynite import FEModel3D
except ImportError:
    import sys
    !{sys.executable} -m pip install -q "PyNiteFEA" "numpy<2.5"
    from Pynite import FEModel3D

L_bay_mm = L_bay*1000; h1_mm = h1*1000; h2_mm = h2*1000
E_Nmm2 = E_rc  # kPa本身數值等同N/mm^2的1000倍? 不對, 需要正確換算
E_Nmm2 = E_rc*1e-3   # 1 kPa = 0.001 N/mm^2? 重新確認: 1kPa=1e3 Pa=1e3 N/m^2
E_Nmm2 = E_rc/1000   # kPa -> N/mm^2: 1 kPa = 0.001 MPa = 0.001 N/mm^2, 所以除以1000
nu = 0.2
G_Nmm2 = E_Nmm2/(2*(1+nu))
Ac_mm2 = Ac*1e6; Ic_mm4 = Ic*1e12
Ab_mm2 = Ab*1e6; Ib_mm4 = Ib*1e12
J_col_mm4 = 2*Ic_mm4; J_beam_mm4 = 2*Ib_mm4

def build_pynite_model():
    model = FEModel3D()
    model.add_node('N1', 0.0, 0.0, 0.0)
    model.add_node('N2', L_bay_mm, 0.0, 0.0)
    model.add_node('N3', 0.0, h1_mm, 0.0)
    model.add_node('N4', L_bay_mm, h1_mm, 0.0)
    model.add_node('N5', 0.0, h1_mm+h2_mm, 0.0)
    model.add_node('N6', L_bay_mm, h1_mm+h2_mm, 0.0)
    model.def_support('N1', True, True, True, True, True, True)
    model.def_support('N2', True, True, True, True, True, True)
    for n in ['N3','N4','N5','N6']:
        model.def_support(n, False, False, True, True, True, False)
    model.add_material('RC', E=E_Nmm2, G=G_Nmm2, nu=nu, rho=2.4e-9)
    model.add_section('ColSec', Ac_mm2, Ic_mm4, Ic_mm4, J_col_mm4)
    model.add_section('BeamSec', Ab_mm2, Ib_mm4, Ib_mm4, J_beam_mm4)
    model.add_member('C1_1F', 'N1', 'N3', 'RC', 'ColSec')
    model.add_member('C2_1F', 'N2', 'N4', 'RC', 'ColSec')
    model.add_member('C1_2F', 'N3', 'N5', 'RC', 'ColSec')
    model.add_member('C2_2F', 'N4', 'N6', 'RC', 'ColSec')
    model.add_member('B_1F',  'N3', 'N4', 'RC', 'BeamSec')
    model.add_member('B_roof','N5', 'N6', 'RC', 'BeamSec')
    return model

model_pn = build_pynite_model()
model_pn.add_node_load('N3', 'FY', -P_1F_extra*1000)
model_pn.add_node_load('N4', 'FY', -P_1F_extra*1000)
model_pn.add_node_load('N5', 'FY', -P_2F*1000)
model_pn.add_node_load('N6', 'FY', -P_2F*1000)
model_pn.add_node_load('N3', 'FX', F1_Y*1000)
model_pn.add_node_load('N5', 'FX', F2_Y*1000)
model_pn.analyze(check_stability=False)

pynite_members = {'C1_1F':'1F左柱', 'C2_1F':'1F右柱', 'C1_2F':'2F左柱', 'C2_2F':'2F右柱',
                   'B_1F':'1F樑', 'B_roof':'屋頂樑'}
pynite_results = {}
print(f"{'構件':<8}{'端':<4}{'Pu(kN)':>10}{'Vu(kN)':>10}{'Mu(kN-m)':>12}")
for mname, label in pynite_members.items():
    m = model_pn.members[mname]
    N_i, N_j = abs(m.axial(0.0))/1000, abs(m.axial(m.L()))/1000
    V_i, V_j = abs(m.shear('Fy',0.0))/1000, abs(m.shear('Fy',m.L()))/1000
    M_i, M_j = abs(m.moment('Mz',0.0))/1e6, abs(m.moment('Mz',m.L()))/1e6
    pynite_results[(label,'i')] = (N_i,V_i,M_i)
    pynite_results[(label,'j')] = (N_j,V_j,M_j)
    print(f"{label:<8}{'i':<4}{N_i:>10.4f}{V_i:>10.4f}{M_i:>12.4f}")
    print(f"{label:<8}{'j':<4}{N_j:>10.4f}{V_j:>10.4f}{M_j:>12.4f}")


構件      端       Pu(kN)    Vu(kN)    Mu(kN-m)
1F左柱    i     281.5470   12.9394     32.1304
1F左柱    j     281.5470   12.9394     13.1574
1F右柱    i     308.8530   12.8986     32.0346
1F右柱    j     308.8530   12.8986     13.1105
2F左柱    i     142.0064    7.9473     11.0312
2F左柱    j     142.0064    7.9473     16.7842
2F右柱    i     153.1936    7.9527     11.0572
2F右柱    j     153.1936    7.9527     16.7774
1F樑     i       4.9459    8.0594     24.1886
1F樑     j       4.9459    8.0594     24.1677
屋頂樑     i       7.9527    5.5936     16.7842
屋頂樑     j       7.9527    5.5936     16.7774


## 跨工具驗證:`PyFEM`——完整除錯過程,不是一次就對上

`PyFEM` 是三個工具裡除錯過程最曲折的一個,誠實記錄完整過程,不是
只留成功結果。

### 第一個坑:`<NodeConstraints>` 語法陷阱,連單一柱子都沒被固定住

先用**單一懸臂柱**(不是完整 `6` 構件構架)這個最小案例定位問題,
發現求解直接矩陣奇異。查 `globdat.dofs.allConstrainedDofs` 才發現
是空的——`.dat` 檔案裡把多個約束塞在同一行用分號分隔
(`u[1]=0.0; v[1]=0.0; ...`),`PyFEM` 解析器不支援這種語法,導致
整個 `<NodeConstraints>` 區塊完全沒被套用,連底端節點都沒被真正
固定住,系統是完全懸空的剛體,矩陣當然奇異。這正是之前 `VL-14`/
`PyFEM_ONBOARDING.md` 就踩過的 `<ExternalForces>` 同一類語法陷阱,
這次沒有推而廣之檢查到 `<NodeConstraints>`,是自己的疏忽。改成
每行一個指定之後,單一懸臂柱才成功求解。

### 第二個坑:單一元素網格下的剪切鎖死

修正約束語法後,單一懸臂柱的位移(`0.003793m`)跟 `Euler-Bernoulli`
理論值(`PL³/3EI=0.005014m`)差 `24.35%`——這正是 `PyFEM_ONBOARDING.md`
記錄過的 `Beam3D`(`Timoshenko` 公式)在粗網格下的剪切鎖死。掃描不同
細分程度確認收斂趨勢:

In [9]:

import sys, os
PYFEM_DIR = os.path.join(os.getcwd(), "PyFEM")
if not os.path.isdir(PYFEM_DIR):
    !git clone -q https://github.com/jjcremmers/PyFEM.git {PYFEM_DIR}
    !{sys.executable} -m pip install -q -e {PYFEM_DIR}
sys.path.insert(0, PYFEM_DIR)

from pyfem.io.InputReader import InputRead
from pyfem.solvers.Solver import Solver

def run_single_column(n_sub):
    lines = ["<Nodes>", "  1 0.0 0.0 0.0 ;"]
    for k in range(1, n_sub+1):
        lines.append(f"  {k+1} 0.0 {h1*k/n_sub} 0.0 ;")
    lines.append("</Nodes>\n<Elements>")
    for k in range(n_sub):
        lines.append(f"  {k+1} \"ColElem\" {k+1} {k+2} ;")
    lines.append("</Elements>\n<NodeConstraints>")
    for dt in ['u','v','w','rx','ry','rz']:
        lines.append(f"  {dt}[1] = 0.0;")
    for k in range(1, n_sub+2):
        for dt in ['w','rx','ry']:
            lines.append(f"  {dt}[{k}] = 0.0;")
    lines.append("</NodeConstraints>\n<ExternalForces>")
    lines.append(f"  u[{n_sub+1}] = 12.9394;")
    lines.append("</ExternalForces>")
    with open('conv_test.dat', 'w') as f:
        f.write('\n'.join(lines))
    with open('conv_test.pro', 'w') as f:
        f.write(f'''input = "conv_test.dat";
ColElem = {{ type = "Beam3D"; E = {E_rc}; G = {E_rc/(2*1.2)}; A = {Ac}; Ix = {Ic}; Iy = {Ic}; J = {2*Ic}; rho = 2.4e-9; }};
solver = {{ type = "NonlinearSolver"; fixedStep = true; maxCycle = 1; tol = 1.0e-8; iterMax = 50; }};
outputModules = [];
''')
    props, globdat = InputRead('conv_test.pro')
    solver = Solver(props, globdat)
    solver.run(props, globdat)
    return globdat.state[6*n_sub]

theory = 12.9394*h1**3/(3*E_rc*Ic)
print(f"理論值(Euler-Bernoulli): {theory:.6f}m\n")
for n_sub in [1, 2, 4, 8]:
    u_tip = run_single_column(n_sub)
    diff = abs(u_tip-theory)/theory*100
    print(f"n_sub={n_sub}: u_tip={u_tip:.6f}m, 差異={diff:.2f}%")


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

    PyFEM analysis: conv_test.pro


  Reading nodes


  ---------------------------------------------------------------------------------


  Number of nodes .................... : 2


...

                                   Converged


      Elapsed time (this step).. : 2.6e-02 sec.


      Total elapsed time........ : 4.4e-02 sec.


  ---------------------------------------------------------------------------------


n_sub=2: u_tip=0.004747m, 差異=5.59%
n_sub=4: u_tip=0.004982m, 差異=0.90%
n_sub=8: u_tip=0.005041m, 差異=0.27%


### 修正後的完整 `6` 構件構架

用 `n_sub=4`(每根構件細分成 `4` 個元素,剪切鎖死殘留約 `1%`)重建
完整構架,語法改成每行一個指定。

In [10]:

n_sub = 4
main_nodes_pf = {1:(0,0), 2:(L_bay,0), 3:(0,h1), 4:(L_bay,h1), 5:(0,h1+h2), 6:(L_bay,h1+h2)}
all_nodes_pf = dict(main_nodes_pf)
next_id = 100
members_pf = []

def build_member_pf(n_start, n_end, sec):
    global next_id
    x0,y0 = all_nodes_pf[n_start]; x1,y1 = all_nodes_pf[n_end]
    node_ids = [n_start]
    for k in range(1, n_sub):
        t = k/n_sub
        nid = next_id; next_id += 1
        all_nodes_pf[nid] = (x0+(x1-x0)*t, y0+(y1-y0)*t)
        node_ids.append(nid)
    node_ids.append(n_end)
    for k in range(n_sub):
        members_pf.append((node_ids[k], node_ids[k+1], sec))

for (a,b,sec) in [(1,3,'col'),(2,4,'col'),(3,5,'col'),(4,6,'col'),(3,4,'beam'),(5,6,'beam')]:
    build_member_pf(a,b,sec)

dat_lines = ["<Nodes>"]
for nid, (x,y) in all_nodes_pf.items():
    dat_lines.append(f"  {nid} {x} {y} 0.0 ;")
dat_lines.append("</Nodes>\n<Elements>")
for i, (ni, nj, sec) in enumerate(members_pf):
    elname = "ColElem" if sec == 'col' else "BeamElem"
    dat_lines.append(f"  {i+1} \"{elname}\" {ni} {nj} ;")
dat_lines.append("</Elements>\n<NodeConstraints>")
for nid in [1,2]:
    for dt in ['u','v','w','rx','ry','rz']:
        dat_lines.append(f"  {dt}[{nid}] = 0.0;")
for nid in all_nodes_pf:
    if nid not in [1,2]:
        for dt in ['w','rx','ry']:
            dat_lines.append(f"  {dt}[{nid}] = 0.0;")
dat_lines.append("</NodeConstraints>\n<ExternalForces>")
dat_lines.append(f"  v[3] = {-P_1F_extra};")
dat_lines.append(f"  v[4] = {-P_1F_extra};")
dat_lines.append(f"  v[5] = {-P_2F};")
dat_lines.append(f"  v[6] = {-P_2F};")
dat_lines.append(f"  u[3] = {F1_Y};")
dat_lines.append(f"  u[5] = {F2_Y};")
dat_lines.append("</ExternalForces>")
with open('canonical_pf.dat', 'w') as f:
    f.write('\n'.join(dat_lines))
with open('canonical_pf.pro', 'w') as f:
    f.write(f'''input = "canonical_pf.dat";
ColElem = {{ type = "Beam3D"; E = {E_rc}; G = {E_rc/(2*1.2)}; A = {Ac}; Ix = {Ic}; Iy = {Ic}; J = {2*Ic}; rho = 2.4e-9; }};
BeamElem = {{ type = "Beam3D"; E = {E_rc}; G = {E_rc/(2*1.2)}; A = {Ab}; Ix = {Ib}; Iy = {Ib}; J = {2*Ib}; rho = 2.4e-9; }};
solver = {{ type = "NonlinearSolver"; fixedStep = true; maxCycle = 1; tol = 1.0e-8; iterMax = 50; }};
outputModules = [];
''')

props_pf, globdat_pf = InputRead('canonical_pf.pro')
solver_pf = Solver(props_pf, globdat_pf)
solver_pf.run(props_pf, globdat_pf)

dofs_pf = globdat_pf.dofs
fint_pf = globdat_pf.fint
total_Fy_pf = fint_pf[dofs_pf.getForType(1,'v')] + fint_pf[dofs_pf.getForType(2,'v')]
total_Fx_pf = fint_pf[dofs_pf.getForType(1,'u')] + fint_pf[dofs_pf.getForType(2,'u')]
print(f"[平衡檢核] 總支撐反力Fy = {total_Fy_pf:.4f}kN (應等於{2*P_1F_extra+2*P_2F:.2f}kN)")
print(f"[平衡檢核] 總支撐反力Fx = {total_Fx_pf:.4f}kN (應等於±{F1_Y+F2_Y:.3f}kN)")
assert abs(total_Fy_pf - (2*P_1F_extra+2*P_2F)) < 1e-4
assert abs(abs(total_Fx_pf) - (F1_Y+F2_Y)) < 1e-4
print("[PASS] PyFEM完整構架平衡檢核通過")

Rx1_pf = fint_pf[dofs_pf.getForType(1,'u')]; Ry1_pf = fint_pf[dofs_pf.getForType(1,'v')]
Rx2_pf = fint_pf[dofs_pf.getForType(2,'u')]; Ry2_pf = fint_pf[dofs_pf.getForType(2,'v')]
print(f"\n1F左柱: Pu={abs(Ry1_pf):.4f}kN, Vu={abs(Rx1_pf):.4f}kN")
print(f"1F右柱: Pu={abs(Ry2_pf):.4f}kN, Vu={abs(Rx2_pf):.4f}kN")


    PyFEM analysis: canonical_pf.pro


  Reading nodes


  ---------------------------------------------------------------------------------


  Number of nodes .................... : 24


  Reading elements


...

                                   Converged


      Elapsed time (this step).. : 6.6e-02 sec.


      Total elapsed time........ : 8.8e-02 sec.


  ---------------------------------------------------------------------------------


[平衡檢核] 總支撐反力Fy = 590.4000kN (應等於590.40kN)
[平衡檢核] 總支撐反力Fx = -25.8380kN (應等於±25.838kN)
[PASS] PyFEM完整構架平衡檢核通過

1F左柱: Pu=281.1563kN, Vu=12.9532kN
1F右柱: Pu=309.2437kN, Vu=12.8848kN


## 四工具最終比對

`OpenSeesPy`/`frame2d`/`PyNite` 三者差異在 `0.02%` 以內;`PyFEM`
差異約 `0.1%~0.14%`,原因明確:`PyFEM` 的 `Beam3D` 是 `Timoshenko`
梁公式(含剪切變形),其他三個工具用的是純 `Euler-Bernoulli`(不含
剪切),即使用 `4` 倍網格細分降低了剪切鎖死,仍殘留一點點方法論
本身的系統性差異——這不是 `bug`,是不同梁理論之間真實存在的差異,
量級完全可以理解。

In [11]:

print(f"{'構件':<8}{'項目':<6}{'Pu/Vu/Mu':>10}")
print(f"1F左柱(i端): OpenSeesPy Pu={opensees_lookup[('1F左柱','i')]['Pu']:.3f}, "
      f"PyNite Pu={pynite_results[('1F左柱','i')][0]:.3f}, "
      f"frame2d Pu=281.546, PyFEM Pu={abs(Ry1_pf):.3f}")
print(f"1F左柱(i端): OpenSeesPy Vu={opensees_lookup[('1F左柱','i')]['Vu']:.3f}, "
      f"PyNite Vu={pynite_results[('1F左柱','i')][1]:.3f}, "
      f"frame2d Vu=12.939, PyFEM Vu={abs(Rx1_pf):.3f}")

diff_pyfem = abs(abs(Ry1_pf)-281.546)/281.546*100
print(f"\nPyFEM跟其他三工具(以frame2d為對照)差異: {diff_pyfem:.3f}%")
print("(方法論差異: Timoshenko vs Euler-Bernoulli, 不是誤差或bug)")


構件      項目      Pu/Vu/Mu
1F左柱(i端): OpenSeesPy Pu=281.547, PyNite Pu=281.547, frame2d Pu=281.546, PyFEM Pu=281.156
1F左柱(i端): OpenSeesPy Vu=12.939, PyNite Vu=12.939, frame2d Vu=12.939, PyFEM Vu=12.953

PyFEM跟其他三工具(以frame2d為對照)差異: 0.138%
(方法論差異: Timoshenko vs Euler-Bernoulli, 不是誤差或bug)
